In [ ]:
from regex_labels.patterns_and_extractors import (
    extract_first_appearance_hearing, determine_bond_status, should_include_ror,
    extract_the_court_appointed_the_orleans_public_defender, extract_ankle_monitoring,
    extract_domestic_stay_away_order, determine_charge_status, extract_gwens_law_hearing,
    has_probable_cause, extract_defendant_in_custody_and_not_in_court,
    extract_the_state_was_not_present, extract_pre_trial_conference,
    determine_continuance, plea_status, guilty_or_no_guilty, sentence_jail_status
)

Config initialized with device: cuda


In [2]:
import pandas as pd
from tqdm.notebook import tqdm

# Load the data
current_clerks_proceedings = "/data/krosen5/cache/transformers/Transformer/proceedings_by_clerk_current.xlsx"
clerks = pd.read_excel(current_clerks_proceedings, sheet_name=None)

# Filter for CDC data only
cdc_dictionary = {}
for clerk, df in clerks.items():
    df_for_clerk = df.copy()
    # Filter the dataframe for identifier column where value is 'cdc'
    cdc_df = df_for_clerk[df_for_clerk['identifier'] == 'cdc']
    cdc_dictionary[clerk] = cdc_df

print(f"Initial CDC clerks: {len(cdc_dictionary)}")

# Remove any clerks from the dictionary where the dataframe is empty
cdc_dictionary = {k: v for k, v in cdc_dictionary.items() if not v.empty}
print(f"CDC clerks with data: {len(cdc_dictionary)}")

Initial CDC clerks: 158
CDC clerks with data: 119


In [3]:
from tqdm.notebook import tqdm

In [4]:
def add_columns_to_dataframe(df):
    df = df.copy()
    
    df['first_appearance_hearing'] = df['text'].apply(extract_first_appearance_hearing)
    df['bond_status'] = df['text'].apply(determine_bond_status)
    df['include_ror'] = df['text'].apply(should_include_ror)
    df['court_appointed_orleans_public_defender'] = df['text'].apply(extract_the_court_appointed_the_orleans_public_defender)
    df['ankle_monitoring'] = df['text'].apply(extract_ankle_monitoring)
    df['domestic_stay_away_order'] = df['text'].apply(extract_domestic_stay_away_order)
    df['charge_status'] = df['text'].apply(determine_charge_status)
    df['gwens_law_hearing'] = df['text'].apply(extract_gwens_law_hearing)
    df['has_probable_cause'] = df['text'].apply(has_probable_cause)
    df['defendant_in_custody_and_not_in_court'] = df['text'].apply(extract_defendant_in_custody_and_not_in_court)
    df['state_was_not_present'] = df['text'].apply(extract_the_state_was_not_present)
    df['pre_trial_conference'] = df['text'].apply(extract_pre_trial_conference)
    df['continuance'] = df['text'].apply(determine_continuance)
    df['plea_status'] = df['text'].apply(plea_status)
    df['guilty_or_no_guilty'] = df['text'].apply(guilty_or_no_guilty)
    df['sentence_jail_status'] = df['text'].apply(sentence_jail_status)

    return df

In [5]:
from tqdm.notebook import tqdm

# Apply feature extraction to each CDC dataframe
print("Processing CDC data...")
for clerk, df in tqdm(cdc_dictionary.items()):
    cdc_dictionary[clerk] = add_columns_to_dataframe(df)

print("CDC processing complete!")

Processing CDC data...


  0%|          | 0/119 [00:00<?, ?it/s]

CDC processing complete!


In [6]:
for clerk, df in tqdm(cdc_dictionary.items()):
    cdc_dictionary[clerk] = df.replace(['NONE', 'NO_DECISION', 'NO_SENTENCE_WAS_IMPOSED'], pd.NA)
    #remove the case_var and date columns
    cdc_dictionary[clerk] = cdc_dictionary[clerk].drop(columns=['case_var', 'date'])
    #drop any duplicate rows
    cdc_dictionary[clerk] = cdc_dictionary[clerk].drop_duplicates()
print("CDC cleaning complete!")


  0%|          | 0/119 [00:00<?, ?it/s]

CDC cleaning complete!


In [7]:
cdc_dictionary_2 = cdc_dictionary.copy()

In [8]:
import re
date_pattern = re.compile(
    r'\b('
        # ISO-like: 2025-09-17 or 2025.09.17 or 2025/09/17
        r'\d{4}[-./]\d{1,2}[-./]\d{1,2}'
        r'|'
        # US-style: 09/16/2025 or 9/6/25
        r'\d{1,2}[-/]\d{1,2}[-/]\d{2,4}'
        r'|'
        # Day first: 17.09.2025 or 17-09-25
        r'\d{1,2}[-.]\d{1,2}[-.]\d{2,4}'
        r'|'
        # Month name first: September 15, 2025
        r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec|January|February|March|April|June|July|August|September|October|November|December)'
        r'\s+\d{1,2},?\s+\d{2,4}'
        r'|'
        # Day first with month name: 15 September 2025 or 15 Sept 25
        r'\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec|January|February|March|April|June|July|August|September|October|November|December)\s+\d{2,4}'
    r')\b',
    flags=re.IGNORECASE
)
def remove_dates(text):
    return date_pattern.sub('[DATE]', text)

cdc_dictionary_2 = {clerk: df.assign(text=df['text'].apply(remove_dates)) for clerk, df in cdc_dictionary_2.items()}



In [9]:
for clerk, df in tqdm(cdc_dictionary_2.items()):
    cdc_dictionary_2[clerk] = cdc_dictionary_2[clerk].drop_duplicates()

  0%|          | 0/119 [00:00<?, ?it/s]

In [10]:
with pd.ExcelWriter('cdc_clerks_with_labels.xlsx') as writer:
    for clerk, df in cdc_dictionary_2.items():
        df.to_excel(writer, sheet_name=clerk, index=False)